In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import io
import time
import random
import math
from robot_client import RobotClient, OscListener
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque, namedtuple

from __future__ import annotations
import random
import argparse
import os
import signal
import sys
from pathlib import Path


import pygame

from src.config import load_config, set_seed
from src.environment import BraitenbergEnv
from src.pygame_view import PygameView

from src.config import load_config, set_seed

pygame 2.6.1 (SDL 2.32.56, Python 3.11.15)
Hello from the pygame community. https://www.pygame.org/contribute.html


<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
c:\Users\LoveTriangle\miniconda3\envs\rl\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
cfg = load_config("config/config.yaml")
width = cfg['arena']['width']
height = cfg['arena']['height']

In [3]:
env = BraitenbergEnv(cfg, cfg.get("condition"))

In [4]:
# if GPU is to be used
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)


In [5]:
device

device(type='cuda')

In [6]:
seed = 42
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

# Model

In [7]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))


class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

In [8]:
class DQN(nn.Module):

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    # Called with either one element to determine next action, or a batch
    # during optimization. Returns tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

In [9]:
  
# def grab_and_build_state(BraitenbergEnv):
#     x = BraitenbergEnv.vehicle.x # need to normalize this
#     y = BraitenbergEnv.vehicle.y
#     x_norm = x/width 
#     y_norm = y/height
#     dt = BraitenbergEnv.step.dt
#     last_x= BraitenbergEnv.last_info.vehicle_x
#     last_y = BraitenbergEnv.last_info.vehicle_y
#     agent_vel = np.sqrt((x-last_x)**2 + (y-last_y)**2)/dt
#     heading_theta = BraitenbergEnv.vehicle.heading
#     sin_theta = np.sin(heading_theta)
#     cos_theta = np.cos(heading_theta)
#     agent_vel = BraitenbergEnv.last_info.
#     red_x, red_y = BraitenbergEnv.red_pos
#     green_x, green_y = BraitenbergEnv.green_pos


#     return np.array([x_norm, y_norm, sin_theta, cos_theta, agent_vel, green_x, green_y,  red_x, red_y])

# Hyperparamters 

In [10]:
# BATCH_SIZE is the number of transitions sampled from the replay buffer
# GAMMA is the discount factor as mentioned in the previous section
# EPS_START is the starting value of epsilon
# EPS_END is the final value of epsilon
# EPS_DECAY controls the rate of exponential decay of epsilon, higher means a slower decay
# TAU is the update rate of the target network
# LR is the learning rate of the ``AdamW`` optimizer

BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.01
EPS_DECAY = 50000
TAU = 0.005
LR = 3e-4


# Get number of actions from gym action space
n_actions = len(env.make_action())
# Get the number of state observations
state,info = env.reset()

n_observations = len(state)

policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10000)


steps_done = 0






In [11]:
def select_action(state):
    global steps_done

    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(
        -steps_done / EPS_DECAY
    )
    steps_done += 1

    if sample > eps_threshold:
        with torch.no_grad():
            action_idx = policy_net(state).max(1).indices.view(1, 1)
            # print("action index:", action_idx.item())
            return action_idx
    else:
        action_idx = random.randrange(len(env.action_space))
        return torch.tensor([[action_idx]], device=device, dtype=torch.long)
    
episode_durations = []
episode_rewards = []

In [12]:
def plot_durations(show_result=False):
    plt.figure(1)
    durations_t = torch.tensor(episode_durations, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Duration')
    plt.plot(durations_t.numpy())
    # Take 100 episode averages and plot them too
    if len(durations_t) >= 100:
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)  # pause a bit so that plots are updated
    # is_ipython = True
    # if is_ipython:
    #     if not show_result:
    #         display.display(plt.gcf())
    #         display.clear_output(wait=True)
    #     else:
    #         display.display(plt.gcf())

In [13]:
def plot_rewards(show_result=False):
    plt.figure(2)
    rewards_t = torch.tensor(episode_rewards, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Total reward')
    plt.plot(rewards_t.numpy())
    # Take 100 episode averages and plot them too
    if len(rewards_t) >= 100:
        means = rewards_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)  # pause a bit so that plots are updated

In [14]:
def run_rollout(policy_net, env, max_steps=None, device=device):
    """Greedy rollout of a trained policy_net in BraitenbergEnv (no exploration)."""
    policy_net.eval()
    max_steps = max_steps or cfg['simulation']['max_steps']

    state, info = env.reset()
    state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    trajectory = {'x': [], 'y': [], 'red_x': [], 'red_y': [], 'green_x': [], 'green_y': []}
    rewards = []

    for t in range(max_steps):
        with torch.no_grad():
            action = policy_net(state_t).max(1).indices.view(1, 1)

        observation, reward, terminated = env.step(action)
        rewards.append(reward)
        trajectory['x'].append(env.vehicle.state.x)
        trajectory['y'].append(env.vehicle.state.y)
        trajectory['red_x'].append(env.red_pos[0])
        trajectory['red_y'].append(env.red_pos[1])
        trajectory['green_x'].append(env.green_pos[0])
        trajectory['green_y'].append(env.green_pos[1])

        if terminated:
            break

        state_t = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

    policy_net.train()
    return trajectory, rewards


def plot_rollout(trajectory, rewards, title_prefix=""):
    fig, ax = plt.subplots()
    ax.plot(trajectory['x'], trajectory['y'], 'o-', color='blue', markersize=3, label='rollout')
    ax.plot(trajectory['x'][0], trajectory['y'][0], 's', color='black', markersize=10, label='start')
    ax.plot(trajectory['x'][-1], trajectory['y'][-1], '*', color='red', markersize=15, label='end')
    ax.plot(trajectory['green_x'][-1], trajectory['green_y'][-1], '^', color='green', markersize=10, label='green target')
    ax.plot(trajectory['red_x'][-1], trajectory['red_y'][-1], 'v', color='darkred', markersize=10, label='red (avoid)')
    half_w = cfg['arena']['width'] / 2
    half_h = cfg['arena']['height'] / 2
    ax.set_xlim(-half_w, half_w)
    ax.set_ylim(half_h, -half_h)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f"{title_prefix}Rollout — total reward: {sum(rewards):.2f}, steps: {len(rewards)}")
    ax.legend()
    plt.show()

# Training

In [15]:
def optimize_model():
    if len(memory) < BATCH_SIZE:
        return
    transitions = memory.sample(BATCH_SIZE)
    # Transpose the batch (see https://stackoverflow.com/a/19343/3343043 for
    # detailed explanation). This converts batch-array of Transitions
    # to Transition of batch-arrays.
    batch = Transition(*zip(*transitions))

    # Compute a mask of non-final states and concatenate the batch elements
    # (a final state would've been the one after which simulation ended)
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                          batch.next_state)), device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state
                                                if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Compute Q(s_t, a) - the model computes Q(s_t), then we select the
    # columns of actions taken. These are the actions which would've been taken
    # for each batch state according to policy_net
    state_action_values = policy_net(state_batch).gather(1, action_batch)

    # Compute V(s_{t+1}) for all next states.
    # Expected values of actions for non_final_next_states are computed based
    # on the "older" target_net; selecting their best reward with max(1).values
    # This is merged based on the mask, such that we'll have either the expected
    # state value or 0 in case the state was final.
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values
    # Compute the expected Q values
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # Compute Huber loss
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    # Optimize the model
    optimizer.zero_grad()
    loss.backward()
    # In-place gradient clipping
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()

In [ ]:
if torch.cuda.is_available() or torch.backends.mps.is_available():
    num_episodes = 10000
else:
    num_episodes = 50

ROLLOUT_EVERY = 10

for i_episode in range(num_episodes):
    # Initialize the environment and get its state
    state, info = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    episode_reward = 0.0
    for t in range(cfg['simulation']['max_steps']):
        action = select_action(state)
        observation, reward, terminated = env.step(action)
        episode_reward += reward
        reward = torch.tensor([reward], device=device)
        done = terminated

        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        # Store the transition in memory
        memory.push(state, action, next_state, reward)

        # Move to the next state
        state = next_state

        # Perform one step of the optimization (on the policy network)
        optimize_model()

        # Soft update of the target network's weights
        # θ′ ← τ θ + (1 −τ )θ′
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*TAU + target_net_state_dict[key]*(1-TAU)
        target_net.load_state_dict(target_net_state_dict)

        if done:
            episode_durations.append(t + 1)
            episode_rewards.append(episode_reward)
            break

    if (i_episode + 1) % ROLLOUT_EVERY == 0:
        plot_rewards()
        rollout_trajectory, rollout_rewards = run_rollout(policy_net, env)
        plot_rollout(rollout_trajectory, rollout_rewards, title_prefix=f"Episode {i_episode + 1} — ")

print('Complete')
plot_rewards(show_result=True)
plt.ioff()
plt.show()

In [17]:
# redo old functions to work with continuous state space




# def get_position(state):
#     """Convert a flat state index into (x, y, orientation).
#     Origin (0, 0) is the top-left corner; y increases downward.
#     State indices run row-major over (x, y), with orientation as the
#     fastest-varying component: state = (y * N_COLS + x) * N_ORIENTATIONS + orientation_idx
#     """
#     total_states = N_COLS * N_ROWS * N_ORIENTATIONS
#     if not (0 <= state < total_states):
#         raise ValueError(f"state {state} is out of bounds for a {N_COLS}x{N_ROWS}x{N_ORIENTATIONS} state space")
#     o_idx = state % N_ORIENTATIONS
#     grid_idx = state // N_ORIENTATIONS
#     x = grid_idx % N_COLS
#     y = grid_idx // N_COLS
#     return x, y, ORIENTATIONS[o_idx]

# Utilities for Real-World

In [19]:
height

240.0

In [ ]:
def run_rollout_REAL(policy_net, env, max_steps=None, device=device, position_timeout=5.0):
    """Greedy rollout of a trained policy_net in BraitenbergEnv (no exploration)."""
    robot = RobotClient()
    robot.start()
    policy_net.eval()
    robot_data = [0, 0, 0, 0]  # [x_red, y_red, x_green, y_green] (raw camera pixels)
    got_position = False

    def on_robot(args):
        nonlocal got_position
        robot_data[:] = args
        got_position = True


    positions = OscListener(port=9000)
    positions.subscribe("/robot", on_robot)
    positions.start()

    waited = 0.0
    while not got_position and waited < position_timeout:
        time.sleep(0.1)
        waited += 0.1
    if not got_position:
        robot.stop()
        raise RuntimeError(f"Timed out after {position_timeout}s waiting for initial /robot position from Bonsai")
    
    if has_invalid_marker(robot_data):
        robot.stop()
        raise RuntimeError("Marker not detected at start — check camera/tracking before running.")
    
    state = robot_to_state(robot_data) # uses real pixel coords to grab state for policy 


    max_steps = max_steps or cfg['simulation']['max_steps']

    # state, info = env.reset() # sim does not happpen 
    state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    trajectory = {'x': [], 'y': [], 'red_x': [], 'red_y': [], 'green_x': [], 'green_y': []}
    rewards = []

    for t in range(max_steps):

        with torch.no_grad():
            action = policy_net(state_t).max(1).indices.view(1, 1)

            left, right, duration = action # dqn actions directly give motor commands
            
            robot.set_wheels(left, right)  # drive
            print(f"State: {state}, Position: ({x}, {y}), Action: {ACTIONS[action]}, Wheels: (L={left}, R={right}), Duration: {duration}s")
            time.sleep(duration) # pauses 

        observation, reward, terminated = env.step(action)
        rewards.append(reward)
    
            
        trajectory['x'].append(env.vehicle.state.x)
        trajectory['y'].append(env.vehicle.state.y)
        trajectory['red_x'].append(env.red_pos[0])
        trajectory['red_y'].append(env.red_pos[1])
        trajectory['green_x'].append(env.green_pos[0])
        trajectory['green_y'].append(env.green_pos[1])

        if terminated:
            break


        state_t = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

    print("Goal reached!" if rewards[-1] == 100 else "Max steps reached without reaching goal.")
    robot.set_wheels(0, 0)
    robot.stop()

    policy_net.train()
    return trajectory, rewards

In [ ]:


def run_policy(start_state, policy, goal_states, max_steps=500, simulation=False, position_timeout=5.0, settle_time=0.2):
    """Roll out `policy` from `start_state` until it reaches the goal region or max_steps.
    When simulation=False, the next state (and the initial state) is read from
    Bonsai's /robot OSC feed (port 9000) instead of the hand-coded transition()
    model and the passed-in start_state — we don't actually know where the robot
    is until we hear from Bonsai, so start_state is only used in simulation mode.
    /robot sends [x_red, y_red, x_green, y_green] — raw camera pixel coordinates
    of the two markers on the robot. See robot_to_state() for how this is converted.

    On real hardware:
    - Before driving, shows a live preview window with the tracked markers and
      goal region overlaid, and blocks on an Enter-to-start prompt so you can
      confirm tracking looks right first.
    - Each step: drive for `duration`, stop the wheels, wait `settle_time` for
      the tracker to catch up, then read the now-stationary position. If a
      marker read comes back invalid (-1, meaning Bonsai lost track of it that
      frame), the state update for that step is skipped and the previous state
      is kept instead of computing garbage from -1 pixel coordinates.
    """
    goal_xy = {get_position(s)[:2] for s in goal_states}

    robot = RobotClient()
    robot.start()

    robot_data = [0, 0, 0, 0]  # [x_red, y_red, x_green, y_green] (raw camera pixels)
    got_position = False

    def on_robot(args):
        nonlocal got_position
        robot_data[:] = args
        got_position = True

    positions = None
    if not simulation:
        positions = OscListener(port=9000)
        positions.subscribe("/robot", on_robot)
        positions.start()

        waited = 0.0
        while not got_position and waited < position_timeout:
            time.sleep(0.1)
            waited += 0.1
        if not got_position:
            robot.stop()
            raise RuntimeError(f"Timed out after {position_timeout}s waiting for initial /robot position from Bonsai")

      
        if has_invalid_marker(robot_data):
            robot.stop()
            raise RuntimeError("Marker not detected at start — check camera/tracking before running.")
        state = robot_to_state(robot_data)
    else:
        state = start_state

    history = [get_position(state)[:2]]
    rewards = []

    for _ in range(max_steps):
        r = reward(state, goal_states)
        rewards.append(r)
        x, y, _ = get_position(state)
        if (x, y) in goal_xy:
            break
        action = policy[state]
        left, right, duration = action_to_motor(action)
        robot.set_wheels(left, right)  # drive
        print(f"State: {state}, Position: ({x}, {y}), Action: {ACTIONS[action]}, Wheels: (L={left}, R={right}), Duration: {duration}s")
        time.sleep(duration) # pauses 
        if simulation:
            state = transition(state, action)
        else:
            robot.set_wheels(0, 0)   # stop moving before reading position
            # time.sleep(settle_time)  # let the tracker catch up to the now-stationary robot
            if has_invalid_marker(robot_data):
                print("Warning: marker not detected this step, keeping previous state.")
            else:
                state = robot_to_state(robot_data)
        history.append(get_position(state)[:2])

    print("Goal reached!" if rewards[-1] > 0 else "Max steps reached without reaching goal.")
    robot.set_wheels(0, 0)
    robot.stop()
    if positions is not None:
        positions.stop()
    return history, rewards


start_state = get_state(x=160, y=200, orientation="N")

trajectory, rewards = run_policy(start_state, test_policy, upper_center_states, simulation=False)
xs, ys = zip(*trajectory)

fig, ax = plt.subplots()
for x, y in {get_position(s)[:2] for s in upper_center_states}:
    ax.scatter(x, y, color="green", s=5, alpha=0.3, label="goal region" if (x, y) == next(iter({get_position(s)[:2] for s in upper_center_states})) else None)
ax.plot(xs, ys, "o-", color="blue", label="rollout")
ax.plot(xs[0], ys[0], "s", color="black", markersize=10, label="start")
ax.plot(xs[-1], ys[-1], "*", color="red", markersize=15, label="end")
ax.set_xlim(0, N_COLS)
ax.set_ylim(N_ROWS, 0)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Rollout — total reward: {sum(rewards)}, steps: {len(rewards)}")
ax.legend()
plt.show()


In [ ]:
# arena corners in raw camera pixel coordinates (calibrated once, camera isn't axis-aligned with the arena)
ARENA_CORNERS_PX = np.array([
    [1217, 400],  # top left
    [157, 890],   # top right
    [1159, 71],   # bottom left
    [179, 172],   # bottom right
], dtype=np.float32)

# corresponding corners in our state-space coordinate frame
ARENA_CORNERS_STATE = np.array([
    [0, 0],
    [height - 1, 0],
    [0, width - 1],
    [height - 1, width - 1],
], dtype=np.float32)

_ARENA_HOMOGRAPHY = cv2.getPerspectiveTransform(ARENA_CORNERS_PX, ARENA_CORNERS_STATE)
_ARENA_HOMOGRAPHY_INV = cv2.getPerspectiveTransform(ARENA_CORNERS_STATE, ARENA_CORNERS_PX)


def pixel_to_arena(px, py):
    """Map a raw camera pixel coordinate to (x, y) in the state-space coordinate frame."""
    pt = np.array([[[px, py]]], dtype=np.float32)
    mapped = cv2.perspectiveTransform(pt, _ARENA_HOMOGRAPHY)
    return float(mapped[0, 0, 0]), float(mapped[0, 0, 1])


def arena_to_pixel(x, y):
    """Map a state-space (x, y) coordinate back to raw camera pixel coordinates
    (the inverse of pixel_to_arena), e.g. for overlaying the goal region on the
    live camera frame.
    """
    pt = np.array([[[x, y]]], dtype=np.float32)
    mapped = cv2.perspectiveTransform(pt, _ARENA_HOMOGRAPHY_INV)
    return float(mapped[0, 0, 0]), float(mapped[0, 0, 1])


def vector_to_orientation(dx, dy):
    """Returns heading direction in radians 0 - 360"""
    if abs(dx) >= abs(dy):
        return "E" if dx > 0 else "W"
    return "S" if dy > 0 else "N"


def has_invalid_marker(robot_data):
    """Bonsai sends -1 for any marker (red/green) it couldn't detect in the current frame."""
    return any(v == -1 for v in robot_data)


def robot_to_state(robot_data):
    """Convert the robot's two-marker camera readout into a  state.
    robot_data: [x_red, y_red, x_green, y_green] in raw camera pixel coordinates.
    Red is mounted on the front of the robot, green on the back; the vector
    from green -> red gives its heading.
    """
    ### need to change bc state is no longer discretized 
    x_red_px, y_red_px, x_green_px, y_green_px = robot_data

    x_red, y_red = pixel_to_arena(x_red_px, y_red_px)
    x_green, y_green = pixel_to_arena(x_green_px, y_green_px)

    orientation = vector_to_orientation(x_red - x_green, y_red - y_green)

    x = min(max(round(x_red), 0), height - 1) # norm happebs according to sim arena parameters, might be an issue
    y = min(max(round(y_red), 0), width - 1)




    return np.array(
            [
                x_norm,
                y_norm,
                sin_theta,
                cos_theta,
                norm_vel,
                norm_green_x,
                norm_green_y,
                norm_red_x,
                norm_red_y,
            ],
            dtype=np.float32,
        )
    

NameError: name 'N_COLS' is not defined

In [ ]:
# aux = [red_x, red_y, green_x, green_y] 